In [0]:
%sql
--1. Load/create data 
CREATE OR REPLACE TABLE marketing_campaigns AS

SELECT
    date,
    campaign_id,
    channel,
    campaign_type,
    audience_segment,
    spend,
    impressions,
    clicks,
    leads,
    mqls,
    opportunities,
    pipeline,
    revenue
FROM VALUES
    ('2026-01-05','C001','Paid Search','Demand Gen','Enterprise',12000,180000,5400,420,180,32,384000,72000),
    ('2026-01-12','C001','Paid Search','Demand Gen','Enterprise',14000,205000,6100,455,195,35,420000,85000),
    ('2026-01-19','C001','Paid Search','Demand Gen','Enterprise',16000,225000,6700,480,205,38,455000,92000),
    ('2026-01-05','C002','Paid Social','Awareness','Enterprise',12000,400000,8000,900,180,12,144000,28000),
    ('2026-01-12','C002','Paid Social','Awareness','Enterprise',14000,470000,9200,1050,205,14,168000,31000),
    ('2026-01-19','C002','Paid Social','Awareness','Enterprise',16000,520000,10500,1200,220,15,180000,34000),
    ('2026-01-05','C003','Webinar','Demand Gen','Mid-Market',10000,80000,4000,500,220,28,280000,65000),
    ('2026-01-12','C003','Webinar','Demand Gen','Mid-Market',10000,85000,4300,540,240,30,300000,72000),
    ('2026-01-19','C003','Webinar','Demand Gen','Mid-Market',10000,90000,4500,575,255,32,320000,76000),
    ('2026-01-05','C004','Partner','Partner Marketing','Enterprise',15000,100000,3500,280,150,30,450000,90000),
    ('2026-01-12','C004','Partner','Partner Marketing','Enterprise',15000,105000,3700,295,158,32,480000,95000),
    ('2026-01-19','C004','Partner','Partner Marketing','Enterprise',15000,110000,3900,310,165,34,510000,102000)
AS t(
    date,
    campaign_id,
    channel,
    campaign_type,
    audience_segment,
    spend,
    impressions,
    clicks,
    leads,
    mqls,
    opportunities,
    pipeline,
    revenue
)

In [0]:
%sql
-- 2. Explore campaign data 
SELECT *
FROM marketing_campaigns
LIMIT 20

In [0]:
%sql
--3. ROI 

SELECT
    campaign_id,
    channel,
    SUM(spend) AS total_spend,
    SUM(leads) AS total_leads,
    SUM(mqls) AS total_mqls,
    SUM(opportunities) AS total_opportunities,
    SUM(pipeline) AS total_pipeline,
    SUM(revenue) AS total_revenue,
    ROUND(SUM(revenue) / SUM(spend), 2) AS ROAS,
    ROUND(SUM(pipeline) / SUM(spend), 2) AS pipeline_to_spend,
    ROUND(SUM(spend) / SUM(leads), 2) AS cost_per_lead,
    ROUND(SUM(spend) / SUM(opportunities), 2) AS cost_per_opportunity
FROM marketing_campaigns
GROUP BY 1,2 
order by roas 

In [0]:
%sql
-- 4. Funnel conversion analysis
SELECT
    campaign_id,
    channel,

    SUM(leads) AS leads,
    SUM(mqls) AS mqls,
    SUM(opportunities) AS opportunities,
    ROUND(SUM(mqls) / SUM(leads) * 100, 2) AS lead_to_mql_pct,
    ROUND(SUM(opportunities) / SUM(mqls) * 100, 2) AS mql_to_opportunity_pct,
    ROUND(SUM(opportunities) / SUM(leads) * 100, 2) AS lead_to_opportunity_pct
FROM marketing_campaigns
GROUP BY 1,2 
ORDER BY lead_to_opportunity_pct DESC;

In [0]:
%sql
-- 5. Funnel economics
-- How efficiently does each campaign generate qualified demand?
SELECT
    campaign_id,
    channel,
    SUM(spend) AS total_spend,
    SUM(mqls) AS total_mqls,
    SUM(opportunities) AS total_opportunities,
    ROUND(SUM(spend) / SUM(mqls), 2) AS cost_per_mql,
    ROUND(SUM(spend) / SUM(opportunities), 2) AS cost_per_opportunity,
    ROUND(SUM(pipeline) / SUM(mqls), 2) AS pipeline_per_mql,
    ROUND(SUM(pipeline) / SUM(opportunities), 2) AS pipeline_per_opportunity

FROM marketing_campaigns
group by 1,2 
ORDER BY cost_per_opportunity;

If we optimized only for lowest cost/opportunity: Webinar.

If we optimized only for pipeline/opportunity: Partner.

If we optimized only for ROAS Webinar

--> shoud not just optimize for one metric 

**stat modeling**

In [0]:
%sql
-- 6A. Create synthetic modeling dataset
-- 30 weeks x 4 channels = 120 observations

CREATE OR REPLACE TABLE marketing_campaigns_model AS
WITH weeks AS (
    SELECT
        explode(
            sequence(
                to_date('2025-01-06'),
                to_date('2025-07-28'),
                interval 7 days
            )
        ) AS date
),

channels AS (
    SELECT * FROM VALUES
        ('Paid Search', 12000, 0.90),
        ('Paid Social', 14000, 0.55),
        ('Webinar',     10000, 1.00),
        ('Partner',     15000, 1.10)
    AS t(channel, base_spend, effectiveness)
),

data AS (
    SELECT
        date,
        channel,

        -- Vary weekly spend around the channel baseline
        ROUND(
            base_spend *
            (0.75 + RAND() * 0.50),
            0
        ) AS spend,

        effectiveness

    FROM weeks
    CROSS JOIN channels
)

SELECT
    date,

    CONCAT(
        'C',
        LPAD(
            CAST(
                ROW_NUMBER() OVER (
                    ORDER BY date, channel
                ) AS STRING
            ),
            4,
            '0'
        )
    ) AS campaign_id,

    channel,

    CASE
        WHEN channel = 'Paid Search' THEN 'Demand Gen'
        WHEN channel = 'Paid Social' THEN 'Awareness'
        WHEN channel = 'Webinar' THEN 'Demand Gen'
        WHEN channel = 'Partner' THEN 'Partner Marketing'
    END AS campaign_type,

    spend,

    -- Generate business outcomes with diminishing returns
    ROUND(
        effectiveness
        * 500000
        * LN(1 + spend / 5000)
        * (0.85 + RAND() * 0.30),
        0
    ) AS pipeline

FROM data;

In [0]:
%sql
SELECT *
FROM marketing_campaigns_model
ORDER BY date, channel
limit 10 

In [0]:
%sql
-- 6B. check we created 120 obs 
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date) AS weeks,
    COUNT(DISTINCT channel) AS channels
FROM marketing_campaigns_model;

In [0]:
%sql
-- 6C. Spend vs. Pipeline -as marketing spend increases, does pipeline generally increase
SELECT
    channel,
    ROUND(AVG(spend), 0) AS avg_weekly_spend,
    ROUND(AVG(pipeline), 0) AS avg_weekly_pipeline,
    ROUND(SUM(pipeline) / SUM(spend), 2) AS pipeline_to_spend
FROM marketing_campaigns_model
GROUP BY channel
ORDER BY pipeline_to_spend DESC;

In [0]:
# 6D. regression - how much does pipeline change when spend increases
# Predict pipeline from marketing spend

from sklearn.linear_model import LinearRegression

# Load our Databricks table into a Pandas DataFrame
df = spark.table("marketing_campaigns_model").toPandas()

# Define X and y
X = df[["spend"]]
y = df["pipeline"]

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Model results
print("Intercept:", round(model.intercept_, 2))
print("Spend coefficient:", round(model.coef_[0], 2))
print("R-squared:", round(model.score(X, y), 3))

so according to 6D, the regressuon model explains only about 6.2% of the variation in pipeline, which is very low. This also means spend alone CANNOT explain pipeline, as there are many things that can affect pipeline: channel, campaign type, audience etc. 

In [0]:
import pandas as pd

df = spark.table("marketing_campaigns_model").toPandas()

df_model = pd.get_dummies(
    df,
    columns=["channel"],
    drop_first=True
)

print(df_model.columns.tolist())

In [0]:
## add channel into the model -- using paid search as the baseline 
# 6E. Regression with channel

from sklearn.linear_model import LinearRegression

# Use the dataframe we created above
X = df_model[
    [
        "spend",
        "channel_Paid Social",
        "channel_Partner",
        "channel_Webinar"
    ]
]

y = df_model["pipeline"]

# Fit the model
model_channel = LinearRegression()
model_channel.fit(X, y)

# Display results
print("Intercept:", round(model_channel.intercept_, 2))

for feature, coefficient in zip(X.columns, model_channel.coef_):
    print(feature, ":", round(coefficient, 2))

print("R-squared:", round(model_channel.score(X, y), 3))

6D response curve 

In [0]:
import numpy as np

In [0]:
df_model["log_spend"] = np.log1p(df_model["spend"] / 5000)

In [0]:
df_model[["spend", "log_spend"]].head(10)


In [0]:
from sklearn.linear_model import LinearRegression

X = df_model[
    [
        "log_spend",
        "channel_Paid Social",
        "channel_Partner",
        "channel_Webinar"
    ]
]

y = df_model["pipeline"]

model_log = LinearRegression()
model_log.fit(X, y)


In [0]:
print("Intercept:", round(model_log.intercept_, 2))

for feature, coefficient in zip(X.columns, model_log.coef_):
    print(feature, ":", round(coefficient, 2))

print("R-squared:", round(model_log.score(X, y), 3))


In [0]:
# 6G. Response Curves
# Part 1: Create hypothetical spend levels

spend_levels = [5000, 10000, 15000, 20000, 25000, 30000]

print(spend_levels)

In [0]:
# Create one scenario for every spend level and channel

channels = [
    "Paid Search",
    "Paid Social",
    "Webinar",
    "Partner"
]

scenarios = []

for channel in channels:
    for spend in spend_levels:
        scenarios.append({
            "channel": channel,
            "spend": spend
        })

scenario_df = pd.DataFrame(scenarios)

scenario_df

In [0]:
# Create the same features used by our regression model
scenario_df["log_spend"] = np.log1p(
    scenario_df["spend"] / 5000
)

scenario_df = pd.get_dummies(
    scenario_df,
    columns=["channel"],
    drop_first=False
)
scenario_df.head(10)


In [0]:
X_scenario = scenario_df[
    [
        "log_spend",
        "channel_Paid Social",
        "channel_Partner",
        "channel_Webinar"
    ]
]

# Predict pipeline
scenario_df["predicted_pipeline"] = model_log.predict(X_scenario)

In [0]:
# Recreate the channel label for easier reporting

scenario_df["channel"] = (
    scenario_df[
        [
            "channel_Paid Search",
            "channel_Paid Social",
            "channel_Partner",
            "channel_Webinar"
        ]
    ]
    .idxmax(axis=1)
    .str.replace("channel_", "", regex=False)
)

# Sort by channel and spend
scenario_df = scenario_df.sort_values(
    ["channel", "spend"]
)

# Calculate incremental pipeline from each additional $5K
scenario_df["incremental_pipeline"] = (
    scenario_df
    .groupby("channel")["predicted_pipeline"]
    .diff()
)

# Show the results
scenario_df[
    [
        "channel",
        "spend",
        "predicted_pipeline",
        "incremental_pipeline"
    ]
].round(0)

channel specific response curve 

In [0]:
# 6H. Channel-specific response curves
# Create interaction terms between channel and log(spend)

for channel in [
    "Paid Social",
    "Partner",
    "Webinar"
]:
    column_name = f"log_spend_{channel.replace(' ', '_')}"
    
    scenario_df[column_name] = (
        scenario_df["log_spend"]
        * scenario_df[f"channel_{channel}"]
    )

scenario_df[
    [
        "channel",
        "spend",
        "log_spend",
        "log_spend_Paid_Social",
        "log_spend_Partner",
        "log_spend_Webinar"
    ]
].head(10)

In [0]:
# 6H. Channel-specific response model

from sklearn.linear_model import LinearRegression

# Create channel × log(spend) interaction terms
df_model["log_spend_Paid_Social"] = (
    df_model["log_spend"] *
    df_model["channel_Paid Social"]
)

df_model["log_spend_Partner"] = (
    df_model["log_spend"] *
    df_model["channel_Partner"]
)

df_model["log_spend_Webinar"] = (
    df_model["log_spend"] *
    df_model["channel_Webinar"]
)

# Define X using the 120-row modeling dataset
X_channel = df_model[
    [
        "log_spend",
        "channel_Paid Social",
        "channel_Partner",
        "channel_Webinar",
        "log_spend_Paid_Social",
        "log_spend_Partner",
        "log_spend_Webinar"
    ]
]

# Define y using the SAME 120 rows
y_channel = df_model["pipeline"]

# Fit the model
model_channel_response = LinearRegression()
model_channel_response.fit(X_channel, y_channel)

# Display results
print("Intercept:", round(model_channel_response.intercept_, 2))

for feature, coefficient in zip(
    X_channel.columns,
    model_channel_response.coef_
):
    print(feature, ":", round(coefficient, 2))

print(
    "R-squared:",
    round(model_channel_response.score(X_channel, y_channel), 3)
)

In [0]:
# 6H. Generate channel-specific pipeline predictions

X_scenario_channel = scenario_df[
    [
        "log_spend",
        "channel_Paid Social",
        "channel_Partner",
        "channel_Webinar",
        "log_spend_Paid_Social",
        "log_spend_Partner",
        "log_spend_Webinar"
    ]
]

# Predict pipeline using the channel-specific model
scenario_df["predicted_pipeline_channel"] = (
    model_channel_response.predict(X_scenario_channel)
)

# Display the results
scenario_df[
    [
        "channel",
        "spend",
        "predicted_pipeline_channel"
    ]
].sort_values(
    ["channel", "spend"]
).round(0)

In [0]:
# 6I. Calculate marginal pipeline and marginal ROI

scenario_df = scenario_df.sort_values(
    ["channel", "spend"]
)

scenario_df["incremental_pipeline_channel"] = (
    scenario_df
    .groupby("channel")["predicted_pipeline_channel"]
    .diff()
)

scenario_df["marginal_pipeline_per_dollar"] = (
    scenario_df["incremental_pipeline_channel"]
    / scenario_df["spend"].diff()
)

scenario_df[
    [
        "channel",
        "spend",
        "predicted_pipeline_channel",
        "incremental_pipeline_channel",
        "marginal_pipeline_per_dollar"
    ]
].round(2)

In [0]:
# 6I. Clean marginal pipeline calculation

scenario_df = scenario_df.sort_values(
    ["channel", "spend"]
)

scenario_df["incremental_pipeline_channel"] = (
    scenario_df
    .groupby("channel")["predicted_pipeline_channel"]
    .diff()
)

scenario_df["marginal_pipeline_per_dollar"] = (
    scenario_df["incremental_pipeline_channel"] / 5000
)

scenario_df[
    [
        "channel",
        "spend",
        "predicted_pipeline_channel",
        "incremental_pipeline_channel",
        "marginal_pipeline_per_dollar"
    ]
].round(2)

In [0]:
# 6J. Budget Optimization
# Step 1: Create possible budget allocations

import itertools
import pandas as pd

budget = 10000

# Possible spend levels for each channel
spend_levels = [5000, 10000, 15000, 20000, 25000, 30000]

channels = [
    "Paid Search",
    "Paid Social",
    "Webinar",
    "Partner"
]

allocations = []

for spends in itertools.product(spend_levels, repeat=4):

    # Only keep allocations that use exactly the total budget
    if sum(spends) == 100000:
        allocations.append({
            "Paid Search": spends[0],
            "Paid Social": spends[1],
            "Webinar": spends[2],
            "Partner": spends[3]
        })

allocation_df = pd.DataFrame(allocations)

print("Number of possible allocations:", len(allocation_df))

allocation_df.head(10)

In [0]:
# 6J. Step 2: Calculate predicted pipeline for each allocation

def predict_pipeline(channel, spend):
    
    # Create model features
    log_spend = np.log1p(spend / 5000)

    paid_social = 1 if channel == "Paid Social" else 0
    partner = 1 if channel == "Partner" else 0
    webinar = 1 if channel == "Webinar" else 0

    # Interaction terms
    log_spend_paid_social = log_spend * paid_social
    log_spend_partner = log_spend * partner
    log_spend_webinar = log_spend * webinar

    features = pd.DataFrame([{
        "log_spend": log_spend,
        "channel_Paid Social": paid_social,
        "channel_Partner": partner,
        "channel_Webinar": webinar,
        "log_spend_Paid_Social": log_spend_paid_social,
        "log_spend_Partner": log_spend_partner,
        "log_spend_Webinar": log_spend_webinar
    }])

    return model_channel_response.predict(features)[0]


# Calculate total predicted pipeline for each allocation

allocation_df["predicted_pipeline"] = (
    allocation_df.apply(
        lambda row:
            predict_pipeline("Paid Search", row["Paid Search"])
            + predict_pipeline("Paid Social", row["Paid Social"])
            + predict_pipeline("Webinar", row["Webinar"])
            + predict_pipeline("Partner", row["Partner"]),
        axis=1
    )
)

# Sort from highest to lowest predicted pipeline

allocation_df = allocation_df.sort_values(
    "predicted_pipeline",
    ascending=False
)

allocation_df.head(10).round(0)